In [0]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import xml.etree.ElementTree as ET
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://poc-sapdevene.enesis.com:8020/sap/opu/odata/sap/ZCDS_LIKP_ODATA_CDS/ZCDS_LIKP_ODATA"
username = dbutils.secrets.get(scope="sap-odata-username", key="client_secret")
password = dbutils.secrets.get(scope="sap-odata-password", key="client_secret")


headers = {
    "Accept": "application/atom+xml",  # eksplisit minta XML
    "sap-client": "120"
}

params = {
    "$filter": "Erdat ge datetime'2019-01-01T00:00:00' and Erdat le datetime'2019-12-31T23:59:59'",
}

response = requests.get(
    url,
    auth=HTTPBasicAuth(username, password),
    headers=headers,
    params=params,
    verify=False,
    timeout=60
)
print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print(f"Response length: {len(response.content)} bytes")
print("---RAW---")
print(response.text)

# Parse XML
try:    root = ET.fromstring(response.content)
except ET.ParseError as e:
    raise RuntimeError(f"Failed to parse XML: {e}")

# Parse XML
root = ET.fromstring(response.content)

ns = {
    "atom": "http://www.w3.org/2005/Atom",
    "m": "http://schemas.microsoft.com/ado/2007/08/dataservices/metadata",
    "d": "http://schemas.microsoft.com/ado/2007/08/dataservices"
}

records = []

# Cek apakah root adalah <feed> (collection) atau <entry> (single)
root_tag = root.tag.split("}")[-1]

if root_tag == "feed":
    # Collection — loop semua entry
    for entry in root.findall("atom:entry", ns):
        props = entry.find(".//m:properties", ns)
        if props is not None:
            row = {child.tag.split("}")[-1]: child.text for child in props}
            records.append(row)

elif root_tag == "entry":
    # Single entity — langsung ambil properties dari root
    props = root.find(".//m:properties", ns)
    if props is not None:
        row = {child.tag.split("}")[-1]: child.text for child in props}
        records.append(row)

df = pd.DataFrame(records)
print(f"Total rows: {len(df):,}")
print(df.T)  # .T biar keliatan semua kolom (karena banyak field)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StringType

# ============================================================
# KNA1 Schema — SAP OData → Databricks Bronze
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    NullType, StringType, DateType, 
    DecimalType, IntegerType, LongType, TimestampType
)
from pyspark.sql.functions import col, to_date, current_timestamp, lit

likp_schema = {
    "Vbeln"                          : StringType(),
    "Ernam"                          : StringType(),
    "Erzet"                          : StringType(),        # TIMS
    "Erdat"                          : DateType(),          # DATS — Created On
    "Bzirk"                          : StringType(),
    "Vstel"                          : StringType(),
    "Vkorg"                          : StringType(),
    "Lfart"                          : StringType(),
    "Autlf"                          : StringType(),
    "Kzazu"                          : StringType(),
    "Wadat"                          : DateType(),          # DATS — Goods Issue Date
    "Lddat"                          : DateType(),          # DATS — Loading Date
    "Tddat"                          : DateType(),          # DATS — Transportation Planning Date
    "Lfdat"                          : DateType(),          # DATS — Delivery Date
    "Kodat"                          : DateType(),          # DATS — Picking Date
    "Ablad"                          : StringType(),
    "Inco1"                          : StringType(),
    "Inco2"                          : StringType(),
    "Expkz"                          : StringType(),
    "Route"                          : StringType(),
    "Faksk"                          : StringType(),
    "Lifsk"                          : StringType(),
    "Vbtyp"                          : StringType(),
    "Knfak"                          : StringType(),
    "Tpqua"                          : StringType(),
    "Tpgrp"                          : StringType(),
    "Lprio"                          : StringType(),        # NUMC
    "Vsbed"                          : StringType(),
    "Kunnr"                          : StringType(),
    "Kunag"                          : StringType(),
    "Kdgrp"                          : StringType(),
    "Stzkl"                          : StringType(),        # DEC — keep string
    "Stzzu"                          : StringType(),        # DEC — keep string
    "Btgew"                          : DecimalType(15, 3),  # QUAN — Gross Weight
    "Ntgew"                          : DecimalType(15, 3),  # QUAN — Net Weight
    "Gewei"                          : StringType(),        # UNIT
    "Volum"                          : DecimalType(15, 3),  # QUAN — Volume
    "Voleh"                          : StringType(),        # UNIT
    "Anzpk"                          : StringType(),        # NUMC
    "Berot"                          : StringType(),
    "Lfuhr"                          : StringType(),        # TIMS
    "Grulg"                          : StringType(),
    "Lstel"                          : StringType(),
    "Tragr"                          : StringType(),
    "Fkarv"                          : StringType(),
    "Fkdat"                          : DateType(),          # DATS
    "Perfk"                          : StringType(),
    "Routa"                          : StringType(),
    "Stafo"                          : StringType(),
    "Kalsm"                          : StringType(),
    "Knumv"                          : StringType(),
    "Waerk"                          : StringType(),        # CUKY
    "Vkbur"                          : StringType(),
    "Vbeak"                          : StringType(),        # DEC — keep string
    "Zukrl"                          : StringType(),
    "Verur"                          : StringType(),
    "Commn"                          : StringType(),
    "Stwae"                          : StringType(),        # CUKY
    "Stcur"                          : StringType(),        # DEC — keep string
    "Exnum"                          : StringType(),
    "Aenam"                          : StringType(),
    "Aedat"                          : DateType(),          # DATS — Last Changed Date
    "Lgnum"                          : StringType(),
    "Lispl"                          : StringType(),
    "Vkoiv"                          : StringType(),
    "Vtwiv"                          : StringType(),
    "Spaiv"                          : StringType(),
    "Fkaiv"                          : StringType(),
    "Pioiv"                          : StringType(),
    "Fkdiv"                          : DateType(),          # DATS
    "Kuniv"                          : StringType(),
    "Kkber"                          : StringType(),
    "Knkli"                          : StringType(),
    "Grupp"                          : StringType(),
    "Sbgrp"                          : StringType(),
    "Ctlpc"                          : StringType(),
    "Cmwae"                          : StringType(),        # CUKY
    "Amtbl"                          : DecimalType(23, 2),  # CURR
    "Bolnr"                          : StringType(),
    "Lifnr"                          : StringType(),
    "Traty"                          : StringType(),
    "Traid"                          : StringType(),
    "Cmfre"                          : DateType(),          # DATS
    "Cmngv"                          : DateType(),          # DATS
    "Xabln"                          : StringType(),
    "Bldat"                          : DateType(),          # DATS
    "WadatIst"                       : DateType(),          # DATS
    "Trspg"                          : StringType(),
    "Tpsid"                          : StringType(),
    "Lifex"                          : StringType(),
    "Ternr"                          : StringType(),
    "KalsmCh"                        : StringType(),
    "Klief"                          : StringType(),
    "Kalsp"                          : StringType(),
    "Knump"                          : StringType(),
    "Netwr"                          : DecimalType(23, 2),  # CURR
    "Aulwe"                          : StringType(),
    "Werks"                          : StringType(),
    "Lcnum"                          : StringType(),
    "Abssc"                          : StringType(),
    "Kouhr"                          : StringType(),        # TIMS
    "Tduhr"                          : StringType(),        # TIMS
    "Lduhr"                          : StringType(),        # TIMS
    "Wauhr"                          : StringType(),        # TIMS
    "Lgtor"                          : StringType(),
    "Lgbzo"                          : StringType(),
    "Akwae"                          : StringType(),        # CUKY
    "Akkur"                          : StringType(),        # DEC — keep string
    "Akprz"                          : StringType(),        # DEC — keep string
    "Proli"                          : StringType(),
    "Xblnr"                          : StringType(),
    "Handle"                         : StringType(),
    "Tsegfl"                         : StringType(),
    "Tsegtp"                         : StringType(),
    "Tzonis"                         : StringType(),
    "Tzonrc"                         : StringType(),
    "ContDg"                         : StringType(),
    "Verursys"                       : StringType(),
    "Kzwab"                          : StringType(),
    "Tcode"                          : StringType(),
    "Vsart"                          : StringType(),
    "Trmtyp"                         : StringType(),
    "Sdabw"                          : StringType(),
    "Vbund"                          : StringType(),
    "Xwoff"                          : StringType(),
    "Dirta"                          : StringType(),
    "Prvbe"                          : StringType(),
    "Folar"                          : StringType(),
    "Podat"                          : DateType(),          # DATS
    "Potim"                          : StringType(),        # TIMS
    "Vganz"                          : IntegerType(),       # INT4
    "Imwrk"                          : StringType(),
    "SpeLoekz"                       : StringType(),
    "SpeLocSeq"                      : StringType(),
    "SpeAccAppSts"                   : StringType(),
    "SpeShpInfSts"                   : StringType(),
    "SpeRetCanc"                     : StringType(),
    "SpeWauhrIst"                    : StringType(),        # TIMS
    "SpeWazoneIst"                   : StringType(),
    "SpeRevVlstk"                    : StringType(),
    "SpeLeScenario"                  : StringType(),
    "SpeOrigSys"                     : StringType(),
    "SpeChngSys"                     : StringType(),
    "SpeGeoroute"                    : StringType(),
    "SpeGeorouteind"                 : StringType(),
    "SpeCarrierInd"                  : StringType(),
    "SpeGtsRel"                      : StringType(),
    "SpeGtsRtCde"                    : StringType(),
    "SpeRelTmstmp"                   : StringType(),        # DEC — timestamp numerik SAP, keep string
    "SpeUnitSystem"                  : StringType(),
    "SpeInvBfrGi"                    : StringType(),
    "SpeQiStatus"                    : StringType(),
    "SpeRedInd"                      : StringType(),
    "Sakes"                          : StringType(),
    "SpeLifexType"                   : StringType(),
    "SpeTtype"                       : StringType(),
    "SpeProNumber"                   : StringType(),
    "LocGuid"                        : StringType(),        # RAW — keep string
    "SpeBillingInd"                  : StringType(),
    "PrinterProfile"                 : StringType(),
    "MsrActive"                      : StringType(),
    "Prtnr"                          : StringType(),        # NUMC
    "StgeLocChange"                  : StringType(),
    "TmCtrlKey"                      : StringType(),
    "DlvSplitInitia"                 : StringType(),
    "DlvVersion"                     : StringType(),        # NUMC
    "Dataaging"                      : DateType(),          # DATS
    "GtsVorpa"                       : StringType(),
    "GtsVornu"                       : StringType(),
    "GtsExpvz"                       : StringType(),
    "GtsPorti"                       : StringType(),
    "ItmExpvz"                       : StringType(),
    "ItmStgbe"                       : StringType(),
    "ItmKzgbe"                       : StringType(),
    "ItmVygid"                       : StringType(),
    "ItmIever"                       : StringType(),
    "ItmStabe"                       : StringType(),
    "ItmKzabe"                       : StringType(),
    "Handoverloc"                    : StringType(),
    "Handoverdate"                   : DateType(),          # DATS
    "Handovertime"                   : StringType(),        # TIMS
    "Handovertzone"                  : StringType(),
    "Bestk"                          : StringType(),
    "Cmpsc"                          : StringType(),
    "Cmpsd"                          : StringType(),
    "Cmpsi"                          : StringType(),
    "Cmpsj"                          : StringType(),
    "Cmpsk"                          : StringType(),
    "CmpsCm"                         : StringType(),
    "CmpsTe"                         : StringType(),
    "Cmgst"                          : StringType(),
    "Fkivk"                          : StringType(),
    "Fkstk"                          : StringType(),
    "Gbstk"                          : StringType(),
    "Hdall"                          : StringType(),
    "Hdals"                          : StringType(),
    "Koquk"                          : StringType(),
    "Kostk"                          : StringType(),
    "Lvstk"                          : StringType(),
    "Pdstk"                          : StringType(),
    "Pkstk"                          : StringType(),
    "SpeTmpid"                       : StringType(),
    "Spstg"                          : StringType(),
    "Trsta"                          : StringType(),
    "Uvall"                          : StringType(),
    "Uvals"                          : StringType(),
    "Uvfak"                          : StringType(),
    "Uvfas"                          : StringType(),
    "Uvpak"                          : StringType(),
    "Uvpas"                          : StringType(),
    "Uvpik"                          : StringType(),
    "Uvpis"                          : StringType(),
    "Uvvlk"                          : StringType(),
    "Uvvls"                          : StringType(),
    "Uvwak"                          : StringType(),
    "Uvwas"                          : StringType(),
    "Vestk"                          : StringType(),
    "Vlstk"                          : StringType(),
    "Wbstk"                          : StringType(),
    "Uvk01"                          : StringType(),
    "Uvk02"                          : StringType(),
    "Uvk03"                          : StringType(),
    "Uvk04"                          : StringType(),
    "Uvk05"                          : StringType(),
    "Uvs01"                          : StringType(),
    "Uvs02"                          : StringType(),
    "Uvs03"                          : StringType(),
    "Uvs04"                          : StringType(),
    "Uvs05"                          : StringType(),
    "xsapmpxlbask"                   : StringType(),
    "Incov"                          : StringType(),
    "Inco2L"                         : StringType(),
    "Inco3L"                         : StringType(),
    "OidExtbol"                      : StringType(),
    "OidMiscdl"                      : StringType(),
    "SitkzDb"                        : StringType(),
    "DummyDeliveryInclEewPs"         : StringType(),
    "xbev1xluleinh"                  : StringType(),        # NUMC
    "xbev1xrpfaess"                  : StringType(),        # DEC — keep string
    "xbev1xrpkist"                   : StringType(),        # DEC — keep string
    "xbev1xrpcont"                   : StringType(),        # DEC — keep string
    "xbev1xrpsonst"                  : StringType(),        # DEC — keep string
    "xbev1xrpflgnr"                  : StringType(),        # NUMC
    "IdtCurEvtloc"                   : StringType(),
    "IdtCurEvtqua"                   : StringType(),
    "IdtCurEvttst"                   : StringType(),        # DEC — timestamp numerik SAP
    "IdtCurEstloc"                   : StringType(),
    "IdtCurEstqua"                   : StringType(),
    "IdtCurEsttst"                   : StringType(),        # DEC — timestamp numerik SAP
    "IdtCurWrkqua"                   : StringType(),
    "IdtPreEvtloc"                   : StringType(),
    "IdtPreEvtqua"                   : StringType(),
    "IdtPreEvttst"                   : StringType(),        # DEC — timestamp numerik SAP
    "IdtPreEstloc"                   : StringType(),
    "IdtPreEstqua"                   : StringType(),
    "IdtPreEsttst"                   : StringType(),        # DEC — timestamp numerik SAP
    "IdtPreWrkqua"                   : StringType(),
    "IdtRefEstloc"                   : StringType(),
    "IdtRefEstqua"                   : StringType(),
    "IdtRefEsttst"                   : StringType(),        # DEC — timestamp numerik SAP
    "IdtFirmLfdat"                   : StringType(),
    "IdtDocnum"                      : StringType(),        # NUMC
    "BorgrGrp"                       : StringType(),
    "Kbnkz"                          : StringType(),
    "FshTransaction"                 : StringType(),
    "FshVasLastItem"                 : StringType(),        # NUMC
    "FshVasCg"                       : StringType(),
    "RfmPsstGroup"                   : StringType(),
}

spark = SparkSession.builder.getOrCreate()

# Step 1: pandas df → Spark df
spark_df = spark.createDataFrame(df)

# Step 2: Cast schema
for col_name, col_type in likp_schema.items():
    if col_name in spark_df.columns:
        if isinstance(col_type, DateType):
            # SAP OData kirim format ISO: "2019-08-23T00:00:00"
            spark_df = spark_df.withColumn(col_name, to_date(col(col_name), "yyyy-MM-dd'T'HH:mm:ss"))
        else:
            spark_df = spark_df.withColumn(col_name, col(col_name).cast(col_type))

# Step 3: Fix sisa kolom NullType
for field in spark_df.schema.fields:
    if isinstance(field.dataType, NullType):
        spark_df = spark_df.withColumn(field.name, spark_df[field.name].cast(StringType()))

# Step 4: Audit columns + write
spark_df = spark_df \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source", lit("ZCDS_LIKP_ODATA"))

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("poc_enesis.bronze.ZCDS_LIKP_ODATA")

print("✅ Bronze table berhasil dibuat!")
spark.sql("SELECT COUNT(*) as total FROM poc_enesis.bronze.ZCDS_LIKP_ODATA").show()

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, to_date

watermark_row = spark.createDataFrame([Row(
    source_name   = "SAP_ODATA",
    table_name    = "ZCDS_LIKP_ODATA",
    last_run_date = RUN_DATE[:10],
    last_run_ts   = None,
    status        = "SUCCESS",
    rows_ingested = int(spark_df.count())
)])

watermark_row = watermark_row \
    .withColumn("last_run_date", to_date("last_run_date")) \
    .withColumn("last_run_ts", current_timestamp())

from delta.tables import DeltaTable
wm = DeltaTable.forName(spark, "poc_enesis.bronze.pipeline_watermark")
wm.alias('t').merge(
    watermark_row.alias('s'),
    't.source_name = s.source_name AND t.table_name = s.table_name'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print("✅ Watermark updated")


In [0]:
import uuid
from datetime import datetime

run_id    = str(uuid.uuid4())[:8]
started   = datetime.utcnow()
SCHEMA     = 'BRONZE'
TABLE     = 'ZCDS_LIKP_ODATA'

try:
    # ... semua logic ingest & write ke bronze ...
    rows_in = spark_df.count()
    status  = 'SUCCESS'
    msg     = f'{rows_in} rows ingested'
except Exception as e:
    rows_in = 0
    status  = 'FAILED'
    msg     = str(e)[:500]
    raise  # re-raise supaya Job tau pipeline gagal
finally:
    ended = datetime.utcnow()
    log_df = spark.createDataFrame([{
        'run_id': run_id, 'schema_name': SCHEMA, 'table_name': TABLE,
        'status': status, 'message': msg, 'rows_in': rows_in, 'rows_out': 0,
        'started_at': started, 'ended_at': ended
    }])
    log_df.write.format('delta').mode('append').saveAsTable(
        'poc_enesis.bronze.run_log'
    )
